In [ ]:
import glob, os
import pandas as pd

folder = r"F:\input_location"          
output = r"F:\output_location\merged_manifest.tsv"  
normalize = False                    

# Read everything as text: person_id never becomes float, empty stays empty, "None" stays "None"
files = sorted(glob.glob(os.path.join(folder, "*.tsv")))
dfs = [pd.read_csv(f, sep="\t", dtype=str, keep_default_na=False) for f in files]

# All manifests must have identical columns
cols = list(dfs[0].columns)
for f, d in zip(files, dfs):
    assert list(d.columns) == cols, f"Different columns in {f}"

df = pd.concat(dfs, ignore_index=True)
n_in = len(df)

path_col = "filepath" if "filepath" in cols else "flow_cube_file_path"
uid_col = "sop_instance_uid" if "sop_instance_uid" in cols else "flow_cube_sop_instance_uid"

# person_id must be digits only
bad = df.loc[~df["person_id"].str.fullmatch(r"\d+"), "person_id"].unique()
assert len(bad) == 0, f"Non-numeric person_id values: {bad[:20]}"

# Remove exact duplicate rows only (never dedupe by UID)
df = df.drop_duplicates()

# Report missing-like values per column
missing_like = ["", "None", "nan", "Nan", "NaN", "null", "Not Provided", "Not reported"]
print("Missing-like values per column:")
for c in cols:
    counts = df.loc[df[c].isin(missing_like), c].value_counts()
    if not counts.empty:
        print(f"  {c}: {counts.to_dict()}")

# Optional normalization
if normalize:
    df = df.replace(missing_like[:-1], "Not reported")

# Sort by person_id numerically, then by file path
df = df.sort_values(
    ["person_id", path_col],
    key=lambda s: s.astype(int) if s.name == "person_id" else s,
    kind="mergesort",
)

df.to_csv(output, sep="\t", index=False)

print(f"{len(files)} files | {n_in} rows in | {n_in - len(df)} duplicates removed | {len(df)} rows out")
print(f"Rows sharing a UID with another row (kept): {df[uid_col].duplicated(keep=False).sum()}")
print(f"Absolute paths: {df[path_col].str.startswith('/').sum()} of {len(df)}")

## fix slashes in filepath

In [ ]:
m = pd.read_csv(r"F:\cirrus\phase1_manifest.tsv", sep="\t", dtype=str, keep_default_na=False)

m["filepath"] = m["filepath"].str.replace("\\", "/", regex=False)
m.to_csv(r"F:\cirrus\phase1_manifest_fixed.tsv", sep="\t", index=False)